# Use the best AutoML generated model to analyze our entire patient cohort

<img src="https://raw.githubusercontent.com/QuentinAmbard/databricks-demo/main/retail/resources/images/lakehouse-retail/lakehouse-retail-churn-ml-experiment.png" style="float: right" width="600px">


Databricks AutoML runs experiments across a grid and creates many models and metrics to determine the best models among all trials. This is a glass-box approach to create a baseline model, meaning we have all the code artifacts and experiments available afterwards. 

Here, we selected the Notebook from the best run from the AutoML experiment.

All the code below has been automatically generated. As data scientists, we can tune it based on our business knowledge, or use the generated model as-is.

This saves data scientists hours of developement and allows team to quickly bootstrap and validate new projects, especally when we may not know the predictors for alternative data such as the telco payment data.


<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F04-Data-Science-ML%2F04.3-Batch-Scoring-patient-readmission&demo_name=lakehouse-hls-readmission&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-hls-readmission%2F04-Data-Science-ML%2F04.3-Batch-Scoring-patient-readmission&version=1">

In [0]:
%pip install mlflow==2.19.0
dbutils.library.restartPython()

  Using cached mlflow-2.19.0-py3-none-any.whl.metadata (30 kB)
  Using cached mlflow_skinny-2.19.0-py3-none-any.whl.metadata (31 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached graphql_relay-3.2.0-py3-none-any.whl.metadata (12 kB)
Using cached mlflow-2.19.0-py3-none-any.whl (27.4 MB)
Using cached mlflow_skinny-2.19.0-py3-none-any.whl (5.9 MB)
Using cached docker-7.1.0-py3-none-any.whl (147 kB)
Using cached graphene-3.4.3-py2.py3-none-any.whl (114 kB)
Using cached graphql_relay-3.2.0-py3-none-any.whl (16 kB)
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 2.21.3
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-dd5e6bf9-77f8-4782-948f-f5ec3cbae8d9
    Can't uninstall 'mlflow-skinny'. No files were found to uninstall.
Note: you may need to restart the

In [0]:
%run ../_resources/00-setup $reset_all_data=false

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_hls_readmission`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


## Running batch inference to score our existing database

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/hls/patient-readmission/patient-risk-ds-flow-3.png?raw=true" width="700px" style="float: right; margin-left: 10px;" />

Our model was created and deployed in production within the MLFlow registry.

We can now easily load it calling the `Production` stage, and use it in any Data Engineering pipeline (a job running every night, in streaming or even within a Spark Declarative Pipelines pipeline).

<br/>

We'll then save this information as a new table so that we can add this information in dashboards or external OLTP databases.

In [0]:
import mlflow
#Make sure we use Mlflow with UC registry
mlflow.set_registry_uri('databricks-uc')
# Load model as a Spark UDF.
loaded_model = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{catalog}.{db}.dbdemos_hls_patient_readmission@prod", result_type='double')

2025/11/03 22:41:01 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 2.19.0, required: mlflow==2.21.3)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025/11/03 22:41:01 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2025/11/03 22:41:01 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


In [0]:
features = loaded_model.metadata.get_input_schema().input_names()

#For this demo, reuse our dataset to test the batch inferences
test_dataset = spark.table('training_dataset')

patient_risk_df =  test_dataset \
                   .withColumn("risk_prediction", loaded_model(struct(*features))) \
                   .select('ENCOUNTER_ID', 'PATIENT_ID', 'risk_prediction')

display(patient_risk_df)

ENCOUNTER_ID,PATIENT_ID,risk_prediction
0003e61d-9662-5111-f6ad-48646551f28c,5646f81d-85f2-c2d2-aa1d-3c30a0413148,0.0
0005e160-77df-b17d-7d70-8d34abca9441,ec3eb7ce-a707-49d3-c67b-758d354b1927,0.0
0006bcbf-7d8c-2dcf-9c01-31cba06578bd,6d241af0-a1b5-6223-1843-b44e39b85659,1.0
0018bc8d-1712-9426-4839-95addc260c48,e1810300-6248-ae6c-d847-4d3a09fdfddb,0.0
001b4d2a-a50b-c76d-d8cc-defa5a6b78e8,a8992104-8997-4c74-3480-3b66fd23df64,1.0
0025e9ef-e36b-ebbe-4b19-72aa70f9a5f6,5646f81d-85f2-c2d2-aa1d-3c30a0413148,0.0
002ab918-4e72-3216-cff6-893b003b978b,bb559f9e-c1ca-8dae-b3d5-12d95505fb98,0.0
002fce7f-eb35-996c-322c-7b587d8cfe2b,4c341ab3-7ecd-3ee6-ab0f-3fb78214d75f,0.0
0034de1c-5e1a-080a-81d1-599aa4443606,f86f92bf-9a5c-2c4e-906d-fe231585d560,0.0
00391881-6f97-9c35-d1d7-fe52b40e78f7,377d69fa-d5e4-085a-cbda-6fabb1fb536e,0.0


In the scored dataframe above, we have essentially created an end-to-end process to predict readmission risk for any patient. 

We have a binary prediction which captures this and incorporates all the intellience from Databricks AutoML and curated features, but this could also return a probability between 0 and 1 depending on how you want your results.

In [0]:
patient_risk_df.write.mode("overwrite").saveAsTable(f"patient_readmission_prediction")


### Next steps

at risk and providing cusom care to reduce readmission risk,
- Deploy Real time inference with [04.4-Model-Serving-patient-readmission]($./04.4-Model-Serving-patient-readmission) to enable realtime capabilities and instantly get insight for a specific patient (Databricks Serverless Model Serving).

Or

- Explain the model for our entire population or a specific patient to understand the risk factors and further personalize care with [04.5-Explainability-patient-readmission]($./04.5-Explainability-patient-readmission)